In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Isso garante que os gráficos apareçam logo abaixo da célula no notebook
%matplotlib inline

In [ ]:
# Lendo o arquivo CSV e transformando em um "DataFrame" (uma tabela do Pandas)
df_orders = pd.read_csv('data/raw/olist_orders_dataset.csv')

In [ ]:
df_orders.head()

In [ ]:
df_orders.info()

In [ ]:
df_orders.isnull().sum()

In [ ]:
# Lista com os nomes de todas as colunas que representam datas
colunas_de_data = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

# Um loop (laço de repetição) para converter todas de uma vez
for coluna in colunas_de_data:
    df_orders[coluna] = pd.to_datetime(df_orders[coluna])

# Rodamos o info() novamente para conferir o resultado
df_orders.info()

In [ ]:
# A função value_counts() conta quantas vezes cada categoria aparece
df_orders['order_status'].value_counts()

In [ ]:
# Criamos um novo DataFrame contendo APENAS os pedidos com status 'delivered'
df_orders_entregues = df_orders[df_orders['order_status'] == 'delivered'].copy()

# Vamos ver quantos nulos sobraram na nossa nova tabela limpa
df_orders_entregues.isnull().sum()

In [ ]:
df_items = pd.read_csv('data/raw/olist_order_items_dataset.csv')
df_customers = pd.read_csv('data/raw/olist_customers_dataset.csv')

# Exibindo as primeiras linhas da tabela de itens para conhecer a estrutura
df_items.head()

In [ ]:
# Unindo pedidos entregues com os itens do pedido
df_pedidos_itens = pd.merge(df_orders_entregues, df_items, on='order_id', how='inner')

# Vamos ver o tamanho da nova tabela
print("Tamanho da tabela original de pedidos:", df_orders_entregues.shape)
print("Tamanho da tabela unida:", df_pedidos_itens.shape)

In [ ]:
# Unindo a tabela anterior com os dados dos clientes
df_master = pd.merge(df_pedidos_itens, df_customers, on='customer_id', how='inner')

# Vamos dar uma olhada na nossa Super Tabela
df_master.head()

In [ ]:
# PAREI AQUI, NÃO LI ESSA PARTE AINDA
# Selecionando apenas as colunas cruciais para a nossa Análise (EDA)
colunas_selecionadas = [
    'order_id', 
    'customer_id', 
    'order_purchase_timestamp', 
    'order_delivered_customer_date', 
    'price', 
    'freight_value', 
    'customer_state', 
    'customer_city'
]

df_master = df_master[colunas_selecionadas]

df_master.info()

In [ ]:
# Criando uma nova coluna apenas com o Ano e o Mês da compra
df_master['mes_ano'] = df_master['order_purchase_timestamp'].dt.to_period('M')

# Agrupando os dados por mês_ano e somando o valor dos itens (price)
faturamento_mensal = df_master.groupby('mes_ano')['price'].sum().reset_index()

# Convertendo o formato de volta para texto para o gráfico renderizar corretamente
faturamento_mensal['mes_ano'] = faturamento_mensal['mes_ano'].astype(str)

faturamento_mensal.head()

# Gráfico 1 - Faturamento ao longo do tempo (Sazonalidade)

In [ ]:
# Configurando o tamanho da figura
plt.figure(figsize=(16, 6))

# Criando o gráfico de linha
sns.lineplot(data=faturamento_mensal, x='mes_ano', y='price', marker='o', color='royalblue')

# Adicionando títulos e rótulos
plt.title('Evolução do Faturamento Mensal (Olist)', fontsize=16, fontweight='bold')
plt.xlabel('Mês e Ano', fontsize=12)
plt.ylabel('Faturamento Total (R$)', fontsize=12)

# Rotacionando as datas no eixo X para facilitar a leitura
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)

# Ajustando o layout para não cortar nada
plt.tight_layout()
plt.show()

### A Sazonalidade do Fim de Ano:
Esse pico em outubro e novembro é o efeito direto do aquecimento para a Black Friday e antecipação de compras de Natal. A queda a partir de maio reflete o esfriamento natural do varejo após o Dia das Mães, um padrão muito clássico no e-commerce.

# Gráfico 2 - Distribuição Geográfica (Top 10 Estados)

In [ ]:
plt.figure(figsize=(12, 6))

# Criando um gráfico de barras horizontais contando o número de clientes por estado
# Usamos value_counts() para pegar os 10 estados com mais ocorrências
sns.countplot(
    data=df_master,
    y='customer_state',
    order=df_master['customer_state'].value_counts().index[:10],
    palette='viridis',
    hue='customer_state',
    legend=False
)

plt.title('Top 10 Estados com Maior Volume de Pedidos', fontsize=16, fontweight='bold')
plt.xlabel('Quantidade de Pedidos Realizados', fontsize=12)
plt.ylabel('Estado (UF)', fontsize=12)

plt.tight_layout()
plt.show()

### A Hegemonia de São Paulo:
A diferença "gritante" de SP faz total sentido não apenas pela densidade populacional, mas pela infraestrutura logística. A maioria dos centros de distribuição fica no Sudeste (especialmente SP), o que torna o frete mais barato e a entrega mais rápida para quem mora lá, incentivando a conversão de vendas.

In [ ]:
# Calculando a diferença em dias entre a entrega e a compra
df_master['tempo_entrega_dias'] = (df_master['order_delivered_customer_date'] - df_master['order_purchase_timestamp']).dt.days

# Vamos ver a média geral de entrega do e-commerce inteiro
media_geral = df_master['tempo_entrega_dias'].mean()
print(f"O tempo médio geral de entrega no Brasil é de: {media_geral:.1f} dias")

# Gráfico 3 - Tempo Médio de Entrega por Estado

In [ ]:
# Agrupando pelo estado e calculando a média de dias de entrega
tempo_medio_estado = df_master.groupby('customer_state')['tempo_entrega_dias'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(14, 6))

# Criando um gráfico de barras. Usamos a paleta 'coolwarm' para destacar os maiores (vermelho) e menores (azul) tempos
sns.barplot(
    data=tempo_medio_estado, 
    x='customer_state', 
    y='tempo_entrega_dias', 
    palette='coolwarm',
    hue='customer_state',
    legend=False
)

# Adicionando uma linha de referência com a média nacional
plt.axhline(media_geral, color='red', linestyle='--', linewidth=2, label=f'Média Nacional ({media_geral:.1f} dias)')

plt.title('Tempo Médio de Entrega por Estado (UF)', fontsize=16, fontweight='bold')
plt.xlabel('Estado (UF)', fontsize=12)
plt.ylabel('Média de Dias para Entrega', fontsize=12)
plt.legend()

plt.tight_layout()
plt.show()

### Análise do Tempo Médio de Entrega por Unidade Federativa no Brasil
A análise do tempo médio de entrega por unidade federativa evidencia diferenças relevantes de desempenho logístico entre os estados brasileiros. Observa-se que São Paulo (SP), apesar de concentrar o maior volume de pedidos do país, apresenta o menor tempo médio de entrega, posicionando-se significativamente abaixo da média nacional (12 dias). Esse resultado sugere elevada eficiência operacional, possivelmente associada a fatores como maior densidade de centros de distribuição, proximidade entre oferta e demanda, infraestrutura de transporte mais desenvolvida e ganhos de escala logísticos.

Adicionalmente, o alto volume de pedidos em SP tende a justificar investimentos mais intensivos em tecnologia, gestão de estoques e otimização de rotas, o que contribui para a redução do lead time logístico. Esse cenário é consistente com modelos de operação em mercados de alta demanda, nos quais a eficiência se torna um diferencial competitivo crítico.

Em contrapartida, estados como Roraima (RR) e Amapá (AP) apresentam os maiores tempos médios de entrega, situando-se bem acima da média nacional. Esse desempenho pode ser explicado por fatores estruturais, como maior distância dos principais centros econômicos e logísticos do país, menor capilaridade da malha de transporte, limitações de infraestrutura e menor densidade de demanda. Esses elementos tendem a aumentar a complexidade operacional e os custos logísticos, impactando negativamente os prazos de entrega.

De forma geral, os dados indicam uma correlação entre eficiência logística, localização geográfica e escala de demanda. Estados mais centrais e economicamente dinâmicos apresentam melhor desempenho logístico, enquanto regiões periféricas enfrentam desafios estruturais que resultam em maiores tempos de entrega.